# Persona Lab — Multi-Agent Synthetic Focus Group
### Applied Generative AI — Dr. Dehghani | Milestone 1 (mock-first skeleton)

**What this is:** a multi-agent system that takes any open-ended idea and runs a panel of
distinct AI personas who *react and disagree*, a **Moderator** that clusters the reactions
into themes, and a **Strategist** that turns the conflict into concrete recommendations.

**Cost-aware routing (3 LLMs):**

| Agent | Job | Model | Why |
|---|---|---|---|
| Persona Panel | high-volume reactions | Gemini Flash-Lite | cheapest, free-tier eligible |
| Moderator | structured JSON clustering | GPT-4o-mini | reliable, cheap JSON |
| Strategist | hardest reasoning step | Claude Haiku | best fit reserved for the key output |

**This milestone runs entirely on MOCKS — zero API calls, $0.** Build and test the whole
pipeline (and later the UI) for free, then flip `USE_MOCKS = False` at Milestone 3 to go live.

## 1. Configuration & cost-aware routing

In [ ]:
USE_MOCKS = True  # M1/M2: keep True (no spend). Flip to False at M3 to use real APIs.

# Each agent routes to the cheapest model that fits its job.
MODEL_ROUTING = {
    "persona":    {"provider": "google",    "model": "gemini-2.5-flash-lite"},
    "moderator":  {"provider": "openai",    "model": "gpt-4o-mini"},
    "strategist": {"provider": "anthropic", "model": "claude-haiku-4-5"},
    # For the FINAL showcase run only, you can bump strategist to a stronger model:
    # "strategist": {"provider": "anthropic", "model": "claude-sonnet-4-6"},
}

import json, time
from concurrent.futures import ThreadPoolExecutor

## 2. Personas
Attitude-based (not demographic, to avoid caricature/bias). Each has *hidden priorities* and a
distinct voice. The system prompt enforces divergence and bans sycophancy — the two failure
modes that would collapse the panel into one agreeable voice.

In [ ]:
PERSONAS = [
    {"name": "Price-Sensitive Skeptic",
     "stance": "Assumes everything is overpriced until proven otherwise.",
     "hidden_priorities": ["lowest total cost", "hidden fees", "free alternatives exist"],
     "voice": "blunt, distrustful of marketing language"},
    {"name": "Early Adopter",
     "stance": "Excited by novelty, tolerates rough edges.",
     "hidden_priorities": ["is it new/different", "bragging rights", "speed of access"],
     "voice": "enthusiastic but easily bored by anything generic"},
    {"name": "Enterprise Buyer",
     "stance": "Cares about ROI, security, and whether it scales to a team.",
     "hidden_priorities": ["data security", "integration cost", "measurable ROI"],
     "voice": "measured, procurement-minded, asks about compliance"},
    {"name": "Risk-Averse Pragmatist",
     "stance": "Defaults to 'why change what works?'",
     "hidden_priorities": ["switching cost", "reliability", "what could go wrong"],
     "voice": "cautious, focuses on failure modes"},
    {"name": "Time-Pressed Generalist",
     "stance": "Will not read instructions; needs value in 30 seconds.",
     "hidden_priorities": ["time to first value", "cognitive load", "do I even get it"],
     "voice": "impatient, judges by first impression"},
]

PERSONA_SYSTEM_TEMPLATE = """You are a member of a synthetic focus group: {name}.
Stance: {stance}
Your private priorities (do NOT state them verbatim): {priorities}
Voice: {voice}

Rules:
- Stay fully in character; your view need NOT agree with the other panelists.
- Be specific. Always include what would make you NOT adopt or buy this.
- Do not be a cheerleader. Sycophancy is failure.

Respond ONLY with JSON (no markdown, no preamble):
{{\"persona\": \"{name}\", \"reaction\": \"<2-3 sentences>\", \"sentiment\": \"positive|mixed|negative\", \"key_objection\": \"<one concrete objection>\"}}"""

MODERATOR_SYSTEM = """You are the Moderator of a focus group. You receive the panel's reactions.
Cluster them into 2-4 themes, note where panelists conflict, identify the single biggest risk.
Respond ONLY with JSON (no markdown, no preamble):
{\"themes\": [{\"theme\": \"<short>\", \"supported_by\": [\"persona names\"], \"tension\": \"<disagreement>\"}],
 \"consensus\": \"<one line>\", \"biggest_risk\": \"<one line>\"}"""

STRATEGIST_SYSTEM = """You are the Strategist. Given the moderator's themes, write concrete,
prioritized recommendations the creator can act on. Be honest about trade-offs. 4-6 bullets max."""

## 3. Mock layer
Canned-but-plausible responses so Milestones 1 and 2 cost nothing. The UI in M2 is built
entirely against these — you only spend money at M3 to verify *quality*, not logic.

In [ ]:
def _mock_persona(name, idea):
    canned = {
        "Price-Sensitive Skeptic": {"reaction": f"'{idea[:40]}...' sounds like something I could rig up myself for free. What exactly am I paying for?", "sentiment": "negative", "key_objection": "No clear reason to pay versus a DIY or free option."},
        "Early Adopter": {"reaction": f"Finally something that isn't another me-too tool. I'd try '{idea[:30]}...' today.", "sentiment": "positive", "key_objection": "If it feels generic after five minutes I'll churn."},
        "Enterprise Buyer": {"reaction": "Interesting, but I can't bring this to my team without knowing where the data goes and the ROI story.", "sentiment": "mixed", "key_objection": "No security/compliance story; unclear ROI."},
        "Risk-Averse Pragmatist": {"reaction": "Our current process works fine. I'd need a strong reason to risk switching.", "sentiment": "negative", "key_objection": "Switching cost and reliability unaddressed."},
        "Time-Pressed Generalist": {"reaction": "I skimmed it and still can't say what it does in one sentence. Make it obvious faster.", "sentiment": "mixed", "key_objection": "Value prop unclear in the first 30 seconds."},
    }
    c = canned.get(name, {"reaction": f"Reaction to {idea[:30]}.", "sentiment": "mixed", "key_objection": "Unclear value."})
    return json.dumps({"persona": name, **c})

def _mock_moderator(_):
    return json.dumps({
        "themes": [
            {"theme": "Unclear value proposition", "supported_by": ["Time-Pressed Generalist", "Price-Sensitive Skeptic"], "tension": "Skeptic wants price justification; Generalist wants instant clarity."},
            {"theme": "Trust & switching cost", "supported_by": ["Enterprise Buyer", "Risk-Averse Pragmatist"], "tension": "Enterprise wants compliance; Pragmatist fears reliability."},
            {"theme": "Novelty appeal", "supported_by": ["Early Adopter"], "tension": "Only the Early Adopter is excited; niche risk."},
        ],
        "consensus": "Everyone agrees the core value must be communicated faster and more concretely.",
        "biggest_risk": "The idea reads as generic, so most personas disengage before seeing the value.",
    })

def _mock_strategist(_):
    return ("- Lead with a one-sentence value prop that lands in under 30 seconds.\n"
            "- Add a concrete 'why pay' contrast vs. free/DIY alternatives.\n"
            "- Publish a short security/data-handling note to unblock enterprise interest.\n"
            "- Offer a low-risk trial path to defuse switching-cost fears.\n"
            "- Keep one bold/novel feature visible to retain early-adopter pull.")

## 4. LLM client (mock ↔ real dispatcher)
Real SDKs are imported *lazily inside each function*, so this notebook runs in mock mode with
nothing installed. At M3: `pip install openai google-genai anthropic`, set the three API keys,
and flip `USE_MOCKS = False`.

In [ ]:
def call_llm(role, system, user, mock_fn, mock_arg):
    if USE_MOCKS:
        time.sleep(0.05)  # simulate latency so M2 streaming work feels realistic
        return mock_fn(mock_arg)
    route = MODEL_ROUTING[role]
    provider, model = route["provider"], route["model"]
    if provider == "google":    return _call_gemini(model, system, user)
    if provider == "openai":    return _call_openai(model, system, user)
    if provider == "anthropic": return _call_anthropic(model, system, user)
    raise ValueError(provider)

# --- Real clients (wired at M3) ---
def _call_gemini(model, system, user):
    from google import genai            # pip install google-genai
    client = genai.Client()             # GEMINI_API_KEY in env
    return client.models.generate_content(model=model, contents=f"{system}\n\n{user}").text

def _call_openai(model, system, user):
    from openai import OpenAI           # pip install openai
    client = OpenAI()                   # OPENAI_API_KEY in env
    r = client.chat.completions.create(model=model, max_tokens=500,
        messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    return r.choices[0].message.content

def _call_anthropic(model, system, user):
    import anthropic                    # pip install anthropic
    client = anthropic.Anthropic()      # ANTHROPIC_API_KEY in env
    r = client.messages.create(model=model, max_tokens=600, system=system,
        messages=[{"role": "user", "content": user}])
    return r.content[0].text

## 5. JSON safety
Models occasionally break JSON or wrap it in markdown fences. This parser strips fences and
retries before falling back, so a bad parse never crashes the demo.

In [ ]:
def _safe_json(raw, fallback, retries=0):
    for _ in range(retries + 1):
        try:
            c = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            return json.loads(c)
        except (json.JSONDecodeError, AttributeError):
            continue
    return fallback

## 6. Agents
The persona panel runs **concurrently** (a `ThreadPoolExecutor`) — important later for latency,
so 5 personas don't run one-after-another in front of the class.

In [ ]:
def _mock_persona_for(name):
    return lambda idea: _mock_persona(name, idea)

def run_persona_panel(idea):
    def one(p):
        system = PERSONA_SYSTEM_TEMPLATE.format(name=p["name"], stance=p["stance"],
            priorities=", ".join(p["hidden_priorities"]), voice=p["voice"])
        raw = call_llm("persona", system, idea, _mock_persona_for(p["name"]), idea)
        return _safe_json(raw, fallback={"persona": p["name"], "reaction": raw,
                          "sentiment": "mixed", "key_objection": "n/a"})
    with ThreadPoolExecutor(max_workers=len(PERSONAS)) as ex:
        return list(ex.map(one, PERSONAS))

def run_moderator(reactions):
    raw = call_llm("moderator", MODERATOR_SYSTEM, json.dumps(reactions), _mock_moderator, reactions)
    return _safe_json(raw, fallback={"themes": [], "consensus": "", "biggest_risk": ""}, retries=2)

def run_strategist(moderation):
    return call_llm("strategist", STRATEGIST_SYSTEM, json.dumps(moderation), _mock_strategist, moderation)

## 7. Orchestrator

In [ ]:
def run_persona_lab(idea):
    reactions = run_persona_panel(idea)
    moderation = run_moderator(reactions)
    strategy = run_strategist(moderation)
    return {"idea": idea, "reactions": reactions, "moderation": moderation, "strategy": strategy}

## 8. Demo run
Run this top-to-bottom. In mock mode it costs nothing and always produces a clean result —
this same seeded run becomes your **demo mode** for rehearsals and the live presentation.

In [ ]:
idea = "A subscription app that turns your grocery receipts into weekly meal plans."
out = run_persona_lab(idea)

print("IDEA:", out["idea"], "\n")
print("=== PANEL ===")
for r in out["reactions"]:
    print(f"[{r['sentiment'].upper():8}] {r['persona']}: {r['reaction']}")
    print(f"           objection: {r['key_objection']}")
print("\n=== MODERATOR ===")
print(json.dumps(out["moderation"], indent=2))
print("\n=== STRATEGIST ===")
print(out["strategy"])

## Next: Milestone 2 (UI, still $0)
Build the Gradio/Streamlit interface against the **mock** pipeline above — persona cards,
progressive streaming as each reaction lands, a themes panel, and a 'demo mode' toggle.
No real calls needed until the interface is done. Then at M3, install the SDKs, set the three
keys, flip `USE_MOCKS = False`, and run a handful of live queries to confirm quality.